# H1 V2b · Building counting asked to our 7B, greedy (plain and with the U-Net fact line)

A follow-up to **H1 V2** (`h1-v2-counts-as-text.ipynb`, kept as it ran). Goal **O2 / KR2**: building damage counting (BDC) above the base model's 34.2%.

**Why:**
- H1 V2 found that greedy answers raise our fine-tuned 7B by 2–6 points on four tasks, and that the U-Net fact line lowered every task it was tried on.
- BDC was never asked to the 7B with greedy answers. Its 499A score, sampled, was already 23.58%, above the U-Net counter of H1 V1 (22.26%).
- Counting is the one task where the fact line is about the question's own subject, so it gets its own test here.

**What it does:** the same model, prompts, image size and greedy settings as H1 V2, on the 4,982 BDC questions only, in two variants:
- **plain:** the official prompt;
- **facts:** the same prompt with H1 V1's fact line before "Question:".

It compares both with the U-Net counter, our 499A score and the base model: overall, per class and per sensor. It also works out the KR2 six-task mean with the best BDC answer, using H1 V2's plain scores for the other five tasks.

**How to run on Kaggle** (the same setup as H1 V2):
1. Settings: **GPU T4 ×2**, internet ON.
2. Inputs: `redwannewaz/disasterm3-bench-mirror-v2` and `h1-v1-facts` (`facts.json`, `H1_v1_summary.csv`). Secret `HF_TOKEN` is optional; the model is public.
3. Run with `SMOKE_TEST = True` first (10 questions per variant, about 10 minutes with the model download). Then Save & Run All, about 3 hours for 9,964 answers. It stops cleanly at 11 hours, and a re-run with this output attached continues where it stopped.
4. Outputs in `/kaggle/working/H1V2b/`: `bdc_results.csv`, `bdc_by_group.csv`, `H1_v2b_per_question.csv`, `config.json`, `README.md`, and `results/building_damage_counting/<tag>/finished.jsonl` for `4_score_results.py`.

**Done when:** the best BDC answer is known (U-Net counter, 7B plain or 7B with facts). It fills the BDC row of the KR2 table; the counting half of KR2 is met only above 34.2%.

In [1]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B model in fp16 needs both T4s"

✓ torch 2.11.0+cu128, transformers 5.16.1, python 3.13.15
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration, the BDC questions, V1's facts and the images ────
import glob
import json
import os
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import pandas as pd

MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"   # our fine-tuned 7B (CSE499A), merged
OUT_DIR = Path("/kaggle/working/H1V2b")
SMOKE_TEST = False             # True = 10 questions per variant
IMAGE_SIZE = 512               # images resized to 512 x 512, as in the 499A evaluation and H1 V2
BATCH = 8                      # questions per generate() call; halved automatically on out-of-memory
MAX_NEW_TOKENS = 32            # answers are option letters
TIME_LIMIT_H = 11.0            # stop cleanly before Kaggle's 12-hour limit
CODE, SUBSET, TASK = "BDC", "building_damage_counting", "Building Damage Counting"   # subset = the 499A scorer's name
VARIANTS = ["plain", "facts"]
CLASS_OF = {"Intact building counting.": "intact", "Damaged building counting.": "damaged",
            "Totally-destroyed building counting.": "destroyed"}
REF = {"Our 7B, 499A (sampled)": 23.58, "Base Qwen2.5-VL-7B": 34.2}
# H1 V2's plain (greedy) scores for the other five KR2 tasks, from its kr2_results.csv (19 Oct)
H1V2_PLAIN = {"DTR": 70.71, "BBR": 20.52, "DRE": 31.86, "DSR": 37.87, "ORR": 22.10}

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_one(pattern, why):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {pattern} under /kaggle/input: {why}")
    return Path(hits[0])

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

with open(find_one("benchmark_release.json", "attach redwannewaz/disasterm3-bench-mirror-v2"), encoding="utf-8") as f:
    bench = json.load(f)
with open(find_one("facts.json", "attach h1-v1-facts (the output of the re-run H1 V1)"), encoding="utf-8") as f:
    FACTS = json.load(f)
v1_hits = sorted(glob.glob("/kaggle/input/**/H1_v1_summary.csv", recursive=True))
V1_SUMMARY = pd.read_csv(v1_hits[0]) if v1_hits else None

bench_index = {}                # every benchmark image by file name: mirrors nest them differently
for p in glob.glob("/kaggle/input/**/test_images/**/*.png", recursive=True):
    bench_index.setdefault(Path(p).name, p)

qs = [(f"{SUBSET}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == TASK)]
questions = qs[:10] if SMOKE_TEST else qs

needed = {name_of(e[k]) for _, e in questions for k in ("pre_image_path", "post_image_path") if e.get(k)}
found = sum(n in bench_index for n in needed)
fact_names = {name_of(e["post_image_path"]).replace("_sar", "") for _, e in questions}
with_fact = sum(n in FACTS for n in fact_names)
if found < 0.99 * len(needed):
    raise FileNotFoundError(f"Only {found:,} of {len(needed):,} benchmark images found: attach the bench mirror.")
if with_fact < 0.99 * len(fact_names):
    raise ValueError(f"facts.json covers {with_fact:,} of {len(fact_names):,} after-images: attach the re-run "
                     "H1 V1 output (its first run wrote an empty facts.json).")
unknown = Counter(e.get("cls_description") for _, e in questions if e.get("cls_description") not in CLASS_OF)
assert not unknown, f"Unexpected BDC classes: {dict(unknown)}"

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ H1 V2b{'  [SMOKE TEST]' if SMOKE_TEST else ''}: {MODEL_ID}")
print(f"  BDC questions: {len(questions):,} {dict(Counter(CLASS_OF[e['cls_description']] for _, e in questions))}")
print(f"  Images found: {found:,} of {len(needed):,}; V1 facts for {with_fact:,} of {len(fact_names):,} after-images")
print(f"  V1 summary (U-Net counter): {'found' if V1_SUMMARY is not None else 'not attached'}")

✓ H1 V2b: AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP
  BDC questions: 4,982 {'intact': 2671, 'damaged': 1345, 'destroyed': 966}
  Images found: 5,443 of 5,443; V1 facts for 2,545 of 2,547 after-images
  V1 summary (U-Net counter): found


In [3]:
# ── Cell 3: The official prompt, the fact line, and the 499A scorer ─────────
from PIL import Image

# Copied from temp_junjue/pyscripts/run_vllm.py (the DisasterM3 authors' benchmark runner):
# building_damage_counting uses the single-choice template with the before and after images, as in H1 V2.
SINGLE_CHOICE = """Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text."""

def fact_for(e):
    """V1's fact line for the question's after-image (SAR questions use the optical twin), or None."""
    f = FACTS.get(name_of(e.get("post_image_path", "")).replace("_sar", ""))
    return f["text"] if f else None

def build(e, fact):
    """-> (chat messages, PIL images in prompt order), built as the official runner builds them."""
    text = SINGLE_CHOICE.format(prompt=e["prompts"], options_str=e["options_str"])
    if fact:
        text = text.replace("Question: ", f"{fact}\n\nQuestion: ", 1)
    parts = text.split("<image>")
    content = [{"type": "text", "text": parts[0]}, {"type": "image"}, {"type": "text", "text": parts[1]},
               {"type": "image"}, {"type": "text", "text": parts[2]}]
    images = [Image.open(bench_index[name_of(e[k])]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
              for k in ("pre_image_path", "post_image_path")]
    return [{"role": "user", "content": content}], images

def letters(text):
    """The 499A scorer (4_score_results.py): the set of capital letters left after removing
    'THE ANSWER IS' and 'OPTION'. An answer counts only if this set equals the truth's set."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

# Self-checks: the scorer, and where the fact line lands
assert letters("B") == {"B"} and letters("The answer is D.") == {"D"} and letters("") == set()
_e = questions[0][1]
_msgs, _imgs = build(_e, "U-Net analysis: TEST.")
_text = "".join(c.get("text", "") for c in _msgs[0]["content"])
assert len(_imgs) == 2 and "U-Net analysis: TEST.\n\nQuestion:" in _text and _text.count("U-Net") == 1
print("✓ Prompt ready (self-checks passed). Example with a fact line:\n")
print(_text[_text.index("U-Net"):][:400])

✓ Prompt ready (self-checks passed). Example with a fact line:

U-Net analysis: TEST.

Question: What is the total number of undamaged buildings following the disaster?
Options: A. 232, B. 324, C. 278, D. 140, E. 186.

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.


In [4]:
# ── Cell 4: Load the model (fp16, split across both T4s) and the batched greedy generator ──
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token())
processor.tokenizer.padding_side = "left"              # batched generation needs left padding
# transformers 4.56 renamed torch_dtype to dtype; older versions may ignore dtype and load fp32 (30 GB)
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, **{dtype_arg: torch.float16}, device_map="auto", attn_implementation="sdpa", token=hf_token())
model.eval()

@torch.inference_mode()
def generate(batch):
    """batch: list of (messages, images) -> list of answers. Greedy, so a re-run gives the same answers."""
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in batch]
    images = [img for _, imgs in batch for img in imgs]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                         top_p=None, top_k=None, repetition_penalty=1.0)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(new_tokens, skip_special_tokens=True)]

def generate_safe(batch):
    """Halves the batch on out-of-memory instead of crashing the run."""
    try:
        return generate(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [""]
        half = len(batch) // 2
        return generate_safe(batch[:half]) + generate_safe(batch[half:])

# One question, plain and with facts, before the long run
_e = questions[0][1]
_answers = generate_safe([build(_e, None), build(_e, fact_for(_e))])
print(f"  BDC: truth {_e['ground_truth_option']!r} | plain {_answers[0]!r} | facts {_answers[1]!r}")
print(f"✓ Model loaded ({dtype_arg}=float16) and checked in {(time.time() - t0) / 60:.1f} min; "
      f"GPU memory {[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

  BDC: truth 'A' | plain 'C' | facts 'C'
✓ Model loaded (dtype=float16) and checked in 2.2 min; GPU memory ['7.6 GB', '9.0 GB']


## Run: every BDC question, plain and then with facts
The plain variant runs first, so if the 11-hour limit stops the run, the plain answers are complete. Answers are appended to `responses/<variant>/building_damage_counting.jsonl` after every batch; a re-run (or a new version with this one's output attached) skips the questions already answered.

In [5]:
# ── Cell 5: Ask every BDC question (plain, then with facts), writing answers as they come ──
from tqdm.auto import tqdm

# Continue an earlier run: copy its answers in when that run's output is attached
previous = sorted(glob.glob("/kaggle/input/**/H1V2b/responses", recursive=True))
if previous and not (OUT_DIR / "responses").exists():
    shutil.copytree(previous[0], OUT_DIR / "responses")
    print(f"Continuing from {previous[0]}")

t_start = time.time()
stopped, unreadable = False, 0
for variant in VARIANTS:
    path = OUT_DIR / "responses" / variant / f"{SUBSET}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    done = {json.loads(line)["id"] for line in open(path, encoding="utf-8")} if path.exists() else set()
    todo = [(qid, e) for qid, e in questions if qid not in done]
    with open(path, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), BATCH), desc=f"BDC {variant}"):
            if (time.time() - t_start) / 3600 > TIME_LIMIT_H:
                stopped = True
                break
            built = []
            for qid, e in todo[i:i + BATCH]:
                try:
                    built.append((qid, build(e, fact_for(e) if variant == "facts" else None)))
                except (KeyError, OSError):              # image missing or unreadable: no answer, scored wrong
                    unreadable += 1
                    f.write(json.dumps({"id": qid, "response": ""}) + "\n")
            answers = generate_safe([b for _, b in built]) if built else []
            for (qid, _), answer in zip(built, answers):
                f.write(json.dumps({"id": qid, "response": answer}) + "\n")
            f.flush()
    if stopped:
        break
run_hours = (time.time() - t_start) / 3600
print(f"{'⚠ Stopped at the time limit' if stopped else '✓ All questions answered'} after {run_hours:.1f} h; "
      f"unreadable images: {unreadable or 'none'}")

BDC plain:   0%|          | 0/623 [00:00<?, ?it/s]

BDC facts:   0%|          | 0/623 [00:00<?, ?it/s]

✓ All questions answered after 2.9 h; unreadable images: 8


In [6]:
# ── Cell 6: Score both variants against the U-Net counter, our 499A score and the base model ──
answers = {}
for variant in VARIANTS:
    path = OUT_DIR / "responses" / variant / f"{SUBSET}.jsonl"
    if path.exists():
        for line in open(path, encoding="utf-8"):
            r = json.loads(line)
            answers[(variant, r["id"])] = r["response"]

rows = []
for qid, e in questions:
    row = {"id": qid, "class": CLASS_OF[e["cls_description"]],
           "sensor": e.get("post_image_type") or e.get("image_type") or "Optical",
           "truth": e["ground_truth_option"]}
    for variant in VARIANTS:
        if (variant, qid) in answers:
            row[variant] = answers[(variant, qid)]
            row[f"{variant}_correct"] = letters(answers[(variant, qid)]) == letters(e["ground_truth_option"])
    rows.append(row)
per_q = pd.DataFrame(rows)
per_q.to_csv(OUT_DIR / "H1_v2b_per_question.csv", index=False)

def pct(df, variant):
    """Accuracy % over the answered questions of one variant (NaN if none were answered)."""
    col = f"{variant}_correct"
    s = df[col].dropna() if col in df else pd.Series(dtype=bool)
    return round(100 * s.astype(bool).mean(), 2) if len(s) else float("nan")

def v1(group):
    """H1 V1's U-Net counter accuracy for one group, from its summary CSV."""
    if V1_SUMMARY is None:
        return float("nan")
    hit = V1_SUMMARY.loc[V1_SUMMARY["group"] == group, "accuracy_%"]
    return float(hit.iloc[0]) if len(hit) else float("nan")

groups = ([("all", per_q)] + [(f"class {c}", per_q[per_q["class"] == c]) for c in ("intact", "damaged", "destroyed")]
          + [(f"{s} after-image", per_q[per_q["sensor"] == s]) for s in ("Optical", "SAR")])
by_group = pd.DataFrame([{"group": name, "questions": len(g), "plain_%": pct(g, "plain"), "facts_%": pct(g, "facts"),
                          "unet_v1_%": v1(name)} for name, g in groups])
by_group.to_csv(OUT_DIR / "bdc_by_group.csv", index=False)
print(by_group.to_string(index=False))

# The best BDC answer, and the KR2 six-task mean with it (H1 V2's plain scores for the other five tasks)
complete = {v: f"{v}_correct" in per_q and bool(per_q[f"{v}_correct"].notna().all()) for v in VARIANTS}
overall = by_group.iloc[0]
candidates = {"U-Net counter (H1 V1)": overall["unet_v1_%"], "7B plain (greedy)": overall["plain_%"],
              "7B with facts (greedy)": overall["facts_%"]}
best_name, best = max(((k, v) for k, v in candidates.items() if pd.notna(v)), key=lambda kv: kv[1])
six_task_mean = round((sum(H1V2_PLAIN.values()) + best) / 6, 2)
results = pd.DataFrame([{"answerer": k, "bdc_%": v} for k, v in {**candidates, **REF}.items()])
results.to_csv(OUT_DIR / "bdc_results.csv", index=False)
verdict = (f"Best BDC answer: {best_name}, {best:.2f}%. KR2 six-task mean with it: {six_task_mean}% "
           f"(base model 31.17%). " + ("The counting half of KR2 is met: above the base model's 34.2%." if best > 34.2
           else f"The counting half of KR2 is not met: {34.2 - best:.2f} points below the base model's 34.2%."))
print(f"\n{results.to_string(index=False)}\n\n{verdict}")
if not all(complete.values()):
    print(f"⚠ Not every question has an answer yet: {complete}. Attach this output and run again to finish.")

# The answers in the 499A format, so 4_score_results.py can score them too
for variant in VARIANTS:
    src = OUT_DIR / "responses" / variant / f"{SUBSET}.jsonl"
    if src.exists():
        tag = f"{MODEL_ID.replace('/', '--')}-greedy" + ("-facts" if variant == "facts" else "")
        dst = OUT_DIR / "results" / SUBSET / tag / "finished.jsonl"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "H1 V2b", "smoke_test": SMOKE_TEST, "model": MODEL_ID, "decoding": "greedy",
          "image_size": IMAGE_SIZE, "batch": BATCH, "max_new_tokens": MAX_NEW_TOKENS,
          "transformers": transformers.__version__, "torch": torch.__version__, "questions": len(questions),
          "complete": complete, "stopped_at_time_limit": stopped, "run_hours": round(run_hours, 2),
          "unreadable_images": unreadable, "best": best_name, "best_bdc_%": best, "six_task_mean_%": six_task_mean}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# H1 V2b: building counting asked to our 7B", "",
    f"Model `{MODEL_ID}`, greedy decoding, images {IMAGE_SIZE} x {IMAGE_SIZE}, the official single-choice prompt"
    f"{' (SMOKE TEST)' if SMOKE_TEST else ''}. `plain` is the official prompt; `facts` adds H1 V1's fact line before "
    "the question.", "",
    "## Results", "", as_markdown(results), "",
    f"**{verdict}**", "",
    "## By class and sensor", "", as_markdown(by_group), "",
    "## Notes", "",
    "- The six-task mean uses H1 V2's plain scores for disaster type, bearing bodies, road damage, land use and "
    "relational reasoning, and the best BDC answer here.",
    "- SAR questions get the fact line of the optical twin of the same tile.", "",
    "## Files", "",
    "- `bdc_results.csv`, `bdc_by_group.csv`, `H1_v2b_per_question.csv`, `config.json`",
    "- `responses/<variant>/building_damage_counting.jsonl` and "
    "`results/building_damage_counting/<tag>/finished.jsonl` (for `4_score_results.py`)",
]), encoding="utf-8")
print(f"✓ Wrote bdc_results.csv, bdc_by_group.csv, H1_v2b_per_question.csv, results/, README.md and config.json to {OUT_DIR}")

              group  questions  plain_%  facts_%  unet_v1_%
                all       4982    30.51    26.60      22.26
       class intact       2671    33.40    24.63      16.66
      class damaged       1345    26.32    27.58      27.36
    class destroyed        966    28.36    30.64      30.64
Optical after-image       3583    32.01    28.10      26.54
    SAR after-image       1399    26.66    22.73      11.29

              answerer  bdc_%
 U-Net counter (H1 V1)  22.26
     7B plain (greedy)  30.51
7B with facts (greedy)  26.60
Our 7B, 499A (sampled)  23.58
    Base Qwen2.5-VL-7B  34.20

Best BDC answer: 7B plain (greedy), 30.51%. KR2 six-task mean with it: 35.6% (base model 31.17%). The counting half of KR2 is not met: 3.69 points below the base model's 34.2%.
✓ Wrote bdc_results.csv, bdc_by_group.csv, H1_v2b_per_question.csv, results/, README.md and config.json to /kaggle/working/H1V2b
